# ASCEND — Dish Classifier v3 (accuracy round: more data + teacher distillation)

Same product contract as v2 — **544 dish classes (85 Indian + 459 world)**,
**MobileNetV4-Conv-M** student, **224px**, quantized **`.tflite`** (INT8 + FP16),
uint8 in/out with normalization folded — trained *better*:

1. **More data per dish:** `data_prep_v3.py` raises the per-class cap
   400 → **1000**, and the lookalike groups v2's eval actually confused
   (naan / dosa / pizza families, wings, rolls, kheer↔phirni, dal family…)
   keep up to **2000**.
2. **Teacher distillation ON:** a **MobileNetV4-Conv-Large** teacher trains
   first on the same data, then the student learns from the teacher's
   softened outputs as well as the hard labels
   (loss = 0.3 · CE + 0.7 · KL · T², T = 3.0).

**Scope note:** same maximum-coverage data decision as v2 (owner's explicit
call, 2026-10-21; audit trail in `DATASETS_V2.md`).

**Before running:** follow `RUNBOOK_V3.md` (GPU on, Internet ON, package
attached). Plan on **2 Kaggle sessions**: session 1 = data prep (~30–60 min)
+ teacher (12 epochs); session 2 = distilled student (15 epochs) + eval +
export. Everything checkpoints, so a session that dies mid-way resumes where
it stopped (Step 2 explains the files).

**Outputs** (in `/kaggle/working/outputs/`): `tflite/indian_food_cls_int8.tflite`
(the shippable v3 model — file stem inherited from v1's converter),
`tflite/indian_food_cls_fp16.tflite`, `labels.txt`, `eval_report.json`
(includes the delta vs v2), `tflite/tflite_eval.json`, plus checkpoints
`teacher.pth` / `teacher_last.pth` / `best_model.pth` / `last_model.pth`.

## Step 0 — Setup

Installs the libraries Kaggle doesn't ship (`timm`, `onnx2tf`, `tf_keras`) and
locates the package files (`data_prep_v3.py`, `convert_tflite.py`,
`taxonomy_v2.json`) — uploaded next to the notebook, fetched from the repo,
or attached as the `ascend-ml-package` input dataset.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

# locate the package files (uploaded next to the notebook, or attached as input dataset)
for name in ("data_prep_v3.py", "convert_tflite.py", "taxonomy_v2.json"):
    if not (WORK / name).exists():
        cands = list(Path("/kaggle/input").rglob(name)) if Path("/kaggle/input").exists() else []
        if cands:
            shutil.copy(cands[0], WORK / name); print(f"found {name} in attached input")
        else:
            print(f"!! {name} NOT FOUND — upload it with this notebook (RUNBOOK step 3)")

# tf_keras is required by onnx2tf's tf_converter backend (convert_tflite.py
# converts via -tb tf_converter); without it that backend fails on Keras 3.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "onnx2tf", "tf_keras"], check=True)

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable GPU in Settings!")

## Step 1 — Data (v3: more per dish, targeted at the confusions)

Two ways in:
- **Path A (default): download in-notebook.** Needs *Internet = On*. Runs
  `data_prep_v3.py`: same six sources as v2 (Khana HF mirror, Food-101
  parquet, FoodX-251 S3, UECFood256, two v1 HF sets), merged per
  `taxonomy_v2.json`, deduped — but the per-class cap is **1000** (was 400)
  and the 41 classes in v2's confusion families (naan/dosa/pizza variants,
  wings, rolls, dal + kheer/phirni, fried snacks…) keep up to **2000**.
  Sources are still merged one at a time and deleted as they merge, so the
  disk-full failure mode of the first v2 attempts stays impossible by
  construction. Expect ~260–330K photos total (v2: ~160K) and 40–70 min
  for this step. A failed source degrades loudly in the report, never silently.
- **Path B: pre-attached data.** If you attached a previous v3 session's
  output (it contains `prepared_v3/labels.txt`), the download is skipped —
  this is the normal session-2 path.

There is no INCLUDE_CONDITIONAL switch: maximum coverage is the point
(the v1 MIT-tagged set is always in, and also contributes burger/pizza).

In [ ]:
import json
PREP = WORK / "data/prepared_v3"

if (PREP / "labels.txt").exists():
    print("using data already in this notebook's working dir")
else:
    attached = list(Path("/kaggle/input").rglob("prepared_v3/labels.txt")) \
        if Path("/kaggle/input").exists() else []
    if attached:
        shutil.copytree(attached[0].parent, PREP)
        print("using attached prepared_v3 data (previous session)")
    else:
        subprocess.run([sys.executable, "data_prep_v3.py", "--out", str(PREP),
                        "--work", str(WORK / "_dl_v3")], check=True, cwd=WORK)

# splitlines, NOT .split(): dish names contain spaces ("pani puri", "vada pav")
# and whitespace-splitting would shred them into phantom classes.
labels = [ln.strip() for ln in (PREP / "labels.txt").read_text().splitlines() if ln.strip()]
report = json.loads((PREP / "prep_report.json").read_text())
regions = json.loads((PREP / "region_map.json").read_text())
n_ind = sum(1 for v in regions.values() if v == "indian")
print(f"{len(labels)} classes ({n_ind} indian / {len(labels) - n_ind} world)")
print("split totals:", report["totals"])
if report.get("source_failures"):
    print("!! SOURCE FAILURES:", report["source_failures"])
print("dropped below floor:", [d["class"] for d in report.get("dropped_classes", [])])

if report.get("target_classes_active"):
    print(f"targeted lookalike classes at cap {report.get('target_cap')}: "
          f"{len(report['target_classes_active'])}")

In [ ]:
# Peek at the data: 2 sample thumbnails per class (first 12 classes)
import matplotlib.pyplot as plt
from PIL import Image
import random
show = labels[:12]
fig, axes = plt.subplots(len(show), 2, figsize=(4, 2 * len(show)))
for r, cls in enumerate(show):
    files = sorted((WORK / "data/prepared_v3/train" / cls).glob("*.*"))
    for c in range(2):
        axes[r][c].imshow(Image.open(files[c * 7 % len(files)]))
        axes[r][c].set_title(cls if c == 0 else "", fontsize=8)
        axes[r][c].axis("off")
plt.tight_layout(); plt.show()

## Step 2 — Train (v3: teacher first, then a distilled student)

**Teacher (new in v3):** `mobilenetv4_conv_large`, ImageNet-pretrained,
**12 epochs**, lr 5e-4, same data/augmentation/sampler as the student. It
exists only to teach — it never ships. Its best-val weights land in
`outputs/teacher.pth`, its resume checkpoint in `outputs/teacher_last.pth`,
and a `teacher_done.txt` marker appears only when all 12 epochs finished —
so a session that dies mid-teacher resumes the teacher, never mistakes a
half-trained one for done.

**Student (the model that ships):** `mobilenetv4_conv_medium`, **15 epochs**,
batch 64, AMP on, cosine schedule — v2's recipe — plus **distillation** from
the frozen teacher:

    loss = 0.3 * CE(label_smoothing=0.1)  +  0.7 * KL(teacher || student) * T^2 ,   T = 3.0

The KL term is computed on temperature-softened logits (the T² factor keeps
its gradient scale comparable to the CE term — the standard Hinton recipe).
The teacher's "dark knowledge" — *this dosa looks 20% like masala dosa* — is
exactly the signal that separates lookalike dishes, which is where v2 lost
most of its accuracy.

**Lookalike hard pairs** also keep v2's 1.5× sampling boost, extended with
the pairs v2's eval surfaced (butter↔garlic naan, paneer↔chicken pizza,
kheer↔phirni, penne↔rigatoni…).

**Checkpoint-resume:** every epoch saves a `*_last.pth` (model + optimizer +
scheduler + best score). If the session ends, attach that version's output
as an input to the next session and Run All — data prep is skipped (Path B),
the teacher is skipped if its done-marker is attached, and whichever model
was mid-training resumes at the next epoch with its schedule intact.

In [ ]:
# ---------- CONFIG ----------
IMG_SIZE = 224
EPOCHS   = 15
BATCH    = 64
LR       = 7e-4
ARCH     = "mobilenetv4_conv_medium"   # the student (ships). fallback if the exporter fights it: "tf_efficientnet_lite0"

# ---- v3 teacher distillation ----
TEACHER_ARCH   = "mobilenetv4_conv_large"
USE_TEACHER    = True
DISTILL        = True                  # student learns from the teacher (needs USE_TEACHER)
TEACHER_EPOCHS = 12
TEACHER_LR     = 5e-4
DISTILL_T      = 3.0                   # softmax temperature
ALPHA_HARD     = 0.3                   # loss = ALPHA_HARD * CE + ALPHA_KD * KL * T^2
ALPHA_KD       = 0.7

SEED = 42
DATA = WORK / "data/prepared_v3"
OUT  = WORK / "outputs"; OUT.mkdir(exist_ok=True)

HARD_PAIRS = [("dal", "dal makhani"), ("dosa", "masala dosa"),
              ("dosa", "idli"), ("biryani", "fried rice"), ("pakora", "samosa"),
              ("naan", "butter naan"), ("spring roll", "egg roll"),
              ("fried rice", "nasi goreng"), ("chicken wings", "buffalo wings"),
              ("omelette", "fried egg"),
              # pairs v2's eval actually produced (count >= 8 in top confusions):
              ("butter naan", "garlic naan"), ("paneer pizza", "chicken pizza"),
              ("pizza", "pepperoni pizza"), ("kheer", "phirni"),
              ("penne", "rigatoni"), ("baby back ribs", "barbecued spareribs"),
              ("sponge cake", "victoria sandwich"), ("rava dosa", "dosa")]
HARD_BOOST = 1.5

In [ ]:
import numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision.transforms as T
import timm
from PIL import Image
from collections import Counter

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

class FolderDataset(Dataset):
    def __init__(self, root, classes, transform):
        self.samples = []
        for ci, cls in enumerate(classes):
            for pat in ("*.jpg", "*.jpeg", "*.png"):
                for f in sorted((root / cls).glob(pat)):
                    self.samples.append((f, ci))
        self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        f, ci = self.samples[i]
        return self.transform(Image.open(f).convert("RGB")), ci

probe = timm.create_model(ARCH, pretrained=False)
cfg = timm.data.resolve_model_data_config(probe)
MEAN, STD = cfg["mean"], cfg["std"]

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.35, contrast=0.3, saturation=0.25, hue=0.02),
    T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

train_ds = FolderDataset(DATA / "train", labels, train_tf)
val_ds   = FolderDataset(DATA / "val",   labels, eval_tf)
test_ds  = FolderDataset(DATA / "test",  labels, eval_tf)

# sampling weights: inverse class frequency x hard-pair boost
counts = Counter(y for _, y in train_ds.samples)
hard_classes = {labels.index(a) for a, b in HARD_PAIRS if a in labels and b in labels} | \
               {labels.index(b) for a, b in HARD_PAIRS if a in labels and b in labels}
weights = []
for _, y in train_ds.samples:
    w = 1.0 / counts[y]
    if y in hard_classes: w *= HARD_BOOST
    weights.append(w)
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_dl = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=2)
val_dl   = DataLoader(val_ds, batch_size=BATCH * 2, num_workers=2)
test_dl  = DataLoader(test_ds, batch_size=BATCH * 2, num_workers=2)
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)} on {device}")

In [ ]:
def evaluate(model, dl):
    model.eval(); logits_all, y_all = [], []
    with torch.no_grad():
        for x, y in dl:
            logits_all.append(model(x.to(device)).cpu().numpy()); y_all.append(y.numpy())
    logits = np.concatenate(logits_all); ys = np.concatenate(y_all)
    order = np.argsort(-logits, axis=1)
    top1 = float((order[:, 0] == ys).mean())
    top5 = float(np.array([ys[i] in order[i, :5] for i in range(len(ys))]).mean())
    return top1, top5, logits, ys

def train_model(model, epochs, lr, dl, val_dl, ckpt_name, teacher=None, resume_from=None,
                last_name="last_model.pth"):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    ce = nn.CrossEntropyLoss(label_smoothing=0.1)
    use_amp = (device == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best, start_ep = -1.0, 0
    if resume_from is not None and Path(resume_from).exists():
        try:
            ck = torch.load(resume_from, map_location=device)
            model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
            sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
            best, start_ep = float(ck["best"]), int(ck["epoch"]) + 1
            print(f"  RESUMING at epoch {start_ep + 1}/{epochs} (best val top1 so far {best:.4f})")
        except Exception as e:
            print(f"  !! checkpoint resume failed ({e}) - training from scratch")
            best, start_ep = -1.0, 0
    for ep in range(start_ep, epochs):
        model.train()
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with torch.autocast("cuda", enabled=use_amp):
                out = model(x)
                loss = ce(out, y)
                if teacher is not None:  # knowledge distillation (Hinton): T=DISTILL_T,
                    # loss = ALPHA_HARD * CE + ALPHA_KD * KL(teacher||student at T) * T^2
                    with torch.no_grad(): t_out = teacher(x)
                    T_ = DISTILL_T
                    kl = nn.functional.kl_div(nn.functional.log_softmax(out.float() / T_, 1),
                                              nn.functional.softmax(t_out.float() / T_, 1),
                                              reduction="batchmean") * T_ * T_
                    loss = ALPHA_HARD * loss + ALPHA_KD * kl
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        v1, v5, _, _ = evaluate(model, val_dl)
        print(f"  epoch {ep+1}/{epochs} val top1={v1:.4f} top5={v5:.4f}")
        if v1 > best:
            best = v1
            torch.save(model.state_dict(), OUT / ckpt_name)
        # last-checkpoint every epoch -> a killed session loses at most 1 epoch.
        # v3: the filename is per-model (last_name) — v2 saved the teacher and the
        # student to the SAME last_model.pth, which would have poisoned resume.
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                    "sched": sched.state_dict(), "scaler": scaler.state_dict(),
                    "epoch": ep, "best": best}, OUT / last_name)
    return best

def find_input_file(name):
    """Return OUT/name, copying it from a previous session's attached output
    first if it is not already in this session's outputs."""
    dest = OUT / name
    if dest.exists():
        return dest
    cands = list(Path("/kaggle/input").rglob(name)) if Path("/kaggle/input").exists() else []
    if cands:
        shutil.copy(cands[0], dest)
        print(f"found {name} in attached input:", cands[0])
        return dest
    return None

# ---------------- teacher (v3) ----------------
# The teacher trains FIRST and only teaches — it never ships. It is skipped
# only when teacher_done.txt is present (written solely after all
# TEACHER_EPOCHS complete); otherwise it resumes from teacher_last.pth.
teacher = None
if USE_TEACHER:
    done_marker = find_input_file("teacher_done.txt")
    if done_marker is not None:
        print("teacher already trained in a previous session — skipping teacher training")
    else:
        t_last = find_input_file("teacher_last.pth")
        print("training teacher:", TEACHER_ARCH)
        t_model = timm.create_model(TEACHER_ARCH, pretrained=True, num_classes=len(labels))
        train_model(t_model, epochs=TEACHER_EPOCHS, lr=TEACHER_LR, dl=train_dl,
                    val_dl=val_dl, ckpt_name="teacher.pth",
                    last_name="teacher_last.pth", resume_from=t_last)
        (OUT / "teacher_done.txt").write_text("ok\n")
        del t_model
        if device == "cuda": torch.cuda.empty_cache()
    if DISTILL:
        find_input_file("teacher.pth")
        teacher = timm.create_model(TEACHER_ARCH, pretrained=False, num_classes=len(labels))
        teacher.load_state_dict(torch.load(OUT / "teacher.pth", map_location=device))
        teacher.to(device).eval()
        for p in teacher.parameters(): p.requires_grad_(False)
        print("teacher loaded frozen for distillation")
    else:
        print("teacher trained for analysis only (DISTILL=False)")

In [ ]:
# ---- resume: pick up a previous session's student checkpoint if attached ----
resume_ckpt = find_input_file("last_model.pth")

print("training student:", ARCH, "(distillation ON)" if teacher is not None else "(plain CE)")
try:
    student = timm.create_model(ARCH, pretrained=True, num_classes=len(labels))
except Exception as e:
    print("primary arch failed, falling back to tf_efficientnet_lite0:", e)
    ARCH = "tf_efficientnet_lite0"
    student = timm.create_model(ARCH, pretrained=True, num_classes=len(labels))
train_model(student, epochs=EPOCHS, lr=LR, dl=train_dl, val_dl=val_dl,
            ckpt_name="best_model.pth", teacher=teacher, resume_from=resume_ckpt)
student.load_state_dict(torch.load(OUT / "best_model.pth"))
print("student done")

## Step 3 — Honest evaluation

Held-out **test** split (never trained on, never used for calibration):
top-1 / top-5 overall, **per-region accuracy (Indian vs world classes)**,
per-class accuracy, and the **top confusion pairs** — the pairs are
the actionable part; they tell you which classes need more photos.

In [ ]:
top1, top5, logits, ys = evaluate(student.to(device), test_dl)
order = np.argsort(-logits, axis=1)
per_class = {}
for ci, cls in enumerate(labels):
    m = ys == ci
    per_class[cls] = round(float((order[m, 0] == ci).mean()), 4) if m.any() else None
conf = np.zeros((len(labels), len(labels)), dtype=int)
for t, p in zip(ys, order[:, 0]): conf[t, p] += 1
pairs = []
for i in range(len(labels)):
    for j in range(len(labels)):
        if i != j and conf[i, j] > 0:
            pairs.append((labels[i], labels[j], int(conf[i, j])))
pairs.sort(key=lambda r: -r[2])
region_of = [regions.get(lbl, "world") for lbl in labels]
per_region = {}
for reg in ("indian", "world"):
    idx = np.array([i for i in range(len(ys)) if region_of[int(ys[i])] == reg])
    if len(idx):
        per_region[reg] = {
            "top1": round(float((order[idx, 0] == ys[idx]).mean()), 4),
            "top5": round(float(np.mean([ys[i] in order[i, :5] for i in idx])), 4),
            "n_test": int(len(idx))}
print("per-region:", per_region)
# v2's verified numbers (models/v2/eval_report.json) — printed so every v3 run
# states plainly whether it actually beat the model it replaces.
V2_BASELINE = {"test_top1": 0.5633, "test_top5": 0.7804,
               "indian_top1": 0.8092, "indian_top5": 0.9497,
               "world_top1": 0.5037, "world_top5": 0.7394}
delta = {"test_top1": round(top1 - V2_BASELINE["test_top1"], 4),
         "test_top5": round(top5 - V2_BASELINE["test_top5"], 4)}
if "indian" in per_region:
    delta["indian_top1"] = round(per_region["indian"]["top1"] - V2_BASELINE["indian_top1"], 4)
print("delta vs v2:", delta)
eval_report = {"model_version": "v3", "arch": ARCH, "img_size": IMG_SIZE, "n_test": int(len(ys)),
               "v2_baseline": V2_BASELINE, "delta_vs_v2": delta,
               "per_region": per_region,
               "test_top1": round(top1, 4), "test_top5": round(top5, 4),
               "per_class_top1": per_class,
               "top_confusions": [{"true": a, "predicted": b, "count": c} for a, b, c in pairs[:10]],
               "labels": labels}
(OUT / "eval_report.json").write_text(json.dumps(eval_report, indent=2))
print(json.dumps({k: eval_report[k] for k in ("test_top1", "test_top5", "n_test")}, indent=2))
print("worst confusions:", pairs[:5])

## Step 4 — Export + quantize

Exports the student to ONNX **with the phone preprocessing baked in**
(input = NHWC float32 raw pixels 0..255 → logits). The normalization affine
(x/255 − mean)/std is **folded into the first conv's weights/bias** before
export, so the graph is a pure CNN behind a single Transpose. Do NOT put
explicit Div/Mul/Add normalization ops in the graph — onnx2tf's tf_converter
backend mangles their constant axes around that Transpose and INT8
calibration dies (the failure post-mortem is in `smoke_train.py`). Export
uses the legacy ONNX exporter (`dynamo=False`): the dynamo exporter's
external weights + Loop node cannot be lowered by onnx2tf. Then this step
runs `convert_tflite.py`: ONNX → SavedModel (onnx2tf, `-tb tf_converter
-kt input`) → **FP32 / FP16 / INT8** TFLite. INT8 is full-integer with a representative set drawn from the
**val** split; it also re-scores every `.tflite` on the held-out test split,
so the size/accuracy trade-off is measured, not assumed.

In [ ]:
# ---- export ONNX: normalization folded into the first conv --------------
# Phone-side contract: input NHWC float32 raw pixels 0..255 -> logits.
# The normalization affine (x/255 - mean)/std is folded into the first
# conv's weights/bias, so the exported graph is a pure CNN behind a
# single Transpose. Every explicit preprocessing-op variant broke in
# onnx2tf's tf_converter backend (post-mortem in smoke_train.py):
# scalar/per-channel Div/Mul/Add constants get their axes mangled around
# the input Transpose and INT8 calibration dies. Folding removes the
# fragile ops entirely; math is identical:
#   y = W * (s*x + t) + b  ==  (W*s) * x + (b + sum_c W_c * t_c)
import copy
export_model = copy.deepcopy(student).cpu().eval()
first_conv = next(m for m in export_model.modules() if isinstance(m, nn.Conv2d))
mean_t, std_t = torch.tensor(MEAN), torch.tensor(STD)
s = 1.0 / (255.0 * std_t)                # per input channel
t = -mean_t / std_t
with torch.no_grad():
    W = first_conv.weight.data           # [O, C, kh, kw]
    b = first_conv.bias.data if first_conv.bias is not None \
        else torch.zeros(W.shape[0])
    new_b = b + (W * t.view(1, -1, 1, 1)).sum(dim=(1, 2, 3))
    first_conv.weight.data = W * s.view(1, -1, 1, 1)
    if first_conv.bias is None:
        first_conv.bias = nn.Parameter(new_b)
    else:
        first_conv.bias.data = new_b

class ExportWrapper(nn.Module):
    """NHWC raw pixels -> permute -> backbone (normalization pre-folded)."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, x_nhwc):                      # [B,H,W,3] float 0..255
        return self.backbone(x_nhwc.permute(0, 3, 1, 2))

wrapped = ExportWrapper(export_model).eval()
# dynamo=False (the legacy exporter) is REQUIRED: the dynamo exporter
# writes external weights and a Loop node that onnx2tf cannot lower.
torch.onnx.export(wrapped, torch.zeros(1, IMG_SIZE, IMG_SIZE, 3), str(OUT / "model.onnx"),
                  opset_version=17, input_names=["input"], output_names=["logits"],
                  dynamic_axes=None, dynamo=False)
(OUT / "labels.txt").write_text("\n".join(labels) + "\n")
print("exported", OUT / "model.onnx")

# convert_tflite.py drives onnx2tf with -tb tf_converter -kt input
# (the tf_converter backend needs tf_keras — installed in Step 0).
r = subprocess.run([sys.executable, "convert_tflite.py", "--onnx", str(OUT / "model.onnx"),
                    "--data-dir", str(DATA), "--out-dir", str(OUT / "tflite"),
                    "--img-size", str(IMG_SIZE)], cwd=WORK, capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-2000:] if r.returncode else "")


## Done — download these from `outputs/`

| File | What it is |
|---|---|
| `tflite/indian_food_cls_int8.tflite` | **The shippable v3 model** (full-integer INT8, uint8 in/out; file stem inherited from v1's converter — the model inside is the 544-class v3) |
| `tflite/indian_food_cls_fp16.tflite` | FP16 fallback (GPU-delegate friendly) |
| `labels.txt` | Class index ↔ dish name |
| `eval_report.json` | FP32 test accuracy overall + **per-region (Indian vs world)** + worst confusion pairs + **delta vs v2** |
| `tflite/tflite_eval.json` | FP32 vs FP16 vs INT8: sizes + test accuracy + agreement |
| `teacher.pth` / `teacher_last.pth` / `last_model.pth` | Checkpoints — only needed to resume an interrupted run in another session |

**Reading the numbers honestly:** v3's exam is the same 544 classes as v2, so
the comparison is direct: v2 scored **0.5633 top-1 / 0.7804 top-5** overall
and **0.8092 / 0.9497** on Indian dishes — `eval_report.json` prints the delta
next to v3's own numbers. The honest expectation for v3 is a few points, not
a leap: roughly **0.58–0.65 top-1 overall, Indian top-1 0.82–0.87**, with the
confusion families (naan, dosa, pizza) improving most. If the delta comes
back near zero or negative, v2 stays the shipping model — the report, not
hope, decides. The app shows top-3 with a confirm step, so top-5 (especially
Indian top-5, already 0.9497) is the number users actually feel.